In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import os
from concurrent.futures import ProcessPoolExecutor
#-------------------------------------

import geopandas as gpd
from keplergl import KeplerGl

#-------------------------------------

import torch
import torchaudio
import IPython.display as ipd
import torchaudio.functional as F
from pydub import AudioSegment
import ast

import librosa
import librosa.display
from tqdm import tqdm

In [2]:
taxotrain = pd.read_csv("EDA_data/taxotrain.csv",index_col=0)

In [4]:
def remove_audio_voices(audio_path, segments_to_remove):
    if isinstance(segments_to_remove, str):
        segments_to_remove = ast.literal_eval(segments_to_remove)
    
    if not segments_to_remove:
        return AudioSegment.from_file("birdclef-2025/train_audio/"+audio_path)
    
    original_audio = AudioSegment.from_file("birdclef-2025/train_audio/"+audio_path)
    
    segments_ms = [
        {
            'start': int(segment['start'] * 1000),
            'end': int(segment['end'] * 1000)
        }
        for segment in segments_to_remove
    ]
    
    sorted_segments = sorted(segments_ms, key=lambda x: x['start'], reverse=True)
    
    modified_audio = original_audio
    for segment in sorted_segments:
        start = segment['start']
        end = segment['end']
        modified_audio = modified_audio[:start] + modified_audio[end:]
    
    return modified_audio

In [5]:
def extract_mfcc(filename, n_mfcc=13):
    audio_path = f"birdclef-2025/train_audio/{filename}"
    modified_audio = remove_audio_voices(filename, taxotrain.loc[taxotrain["filename"] == filename, "human_voices"].values[0])
    
    samples = np.array(modified_audio.get_array_of_samples(), dtype=np.float32) / 32768.0
    
    return librosa.feature.mfcc(y=samples, sr=modified_audio.frame_rate, n_mfcc=n_mfcc)

In [6]:
mfccs = []
with ProcessPoolExecutor() as executor:
    mfccs = list(tqdm(executor.map(extract_mfcc, taxotrain["filename"]), 
                      total=len(taxotrain["filename"]), 
                      desc="Extrayendo MFCCs"))

Extrayendo MFCCs: 100%|██████████| 28564/28564 [1:38:56<00:00,  4.81it/s]  


In [7]:
mfccs

[array([[400.95648  , 400.95648  , 400.95648  , ..., 404.19644  ,
         549.5202   , 630.02203  ],
        [  0.       ,   0.       ,   0.       , ...,   4.578165 ,
          32.65054  ,  20.480347 ],
        [  0.       ,   0.       ,   0.       , ...,   4.56652  ,
         -16.28578  , -17.635855 ],
        ...,
        [  0.       ,   0.       ,   0.       , ...,   4.228652 ,
           3.9987803,   7.4843464],
        [  0.       ,   0.       ,   0.       , ...,   4.163192 ,
           8.325024 ,  10.435759 ],
        [  0.       ,   0.       ,   0.       , ...,   4.0947294,
           7.164385 ,   9.899529 ]], dtype=float32),
 array([[ 3.5082526e+02,  3.5082526e+02,  3.5082526e+02, ...,
          4.6024515e+02,  4.9737939e+02,  4.5017047e+02],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          6.6836319e+01,  6.9181793e+01,  5.7513454e+01],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          1.3391090e+01, -1.4098554e+00,  1.7074896e+00]

In [8]:
taxotrain["mfccs_noise"] = mfccs

In [9]:
taxotrain.to_csv("EDA_data/taxotrainmfccs")

In [10]:
taxotrain

,primary_label,secondary_labels,type,filename,collection,rating,latitude,longitude,scientific_name,common_name,genus,species,inat_taxon_id,class_name,human_voices,mfccs_noise
0,1139490,[''],[''],1139490/CSA36385.ogg,CSA,0.0,7.3206,-73.7128,Ragoniella pulchella,Ragoniella pulchella,Ragoniella,pulchella,1139490,Insecta,"[{'start': 9.1, 'end': 10.5}, {'start': 10.6, ...","[[400.95648, 400.95648, 400.95648, 400.95648, ..."
1,1139490,[''],[''],1139490/CSA36389.ogg,CSA,0.0,7.3206,-73.7128,Ragoniella pulchella,Ragoniella pulchella,Ragoniella,pulchella,1139490,Insecta,"[{'start': 9.9, 'end': 14.4}, {'start': 14.7, ...","[[350.82526, 350.82526, 350.82526, 350.82526, ..."
2,1192948,[''],[''],1192948/CSA36358.ogg,CSA,0.0,7.3791,-73.7313,Oxyprora surinamensis,Oxyprora surinamensis,Oxyprora,surinamensis,1192948,Insecta,"[{'start': 9.7, 'end': 12.3}, {'start': 12.7, ...","[[331.46136, 332.20923, 331.62955, 331.3303, 3..."
3,1192948,[''],[''],1192948/CSA36366.ogg,CSA,0.0,7.2800,-73.8582,Oxyprora surinamensis,Oxyprora surinamensis,Oxyprora,surinamensis,1192948,Insecta,"[{'start': 9.8, 'end': 11.4}, {'start': 11.6, ...","[[432.83264, 432.83264, 432.83264, 432.83264, ..."
4,1192948,[''],[''],1192948/CSA36373.ogg,CSA,0.0,7.3791,-73.7313,Oxyprora surinamensis,Oxyprora surinamensis,Oxyprora,surinamensis,1192948,Insecta,"[{'start': 9.7, 'end': 15.4}, {'start': 15.8, ...","[[403.15134, 403.15134, 403.15134, 403.15134, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28559,ywcpar,[''],[''],ywcpar/iNat77392.ogg,iNat,0.0,7.6921,-80.3379,Amazona ochrocephala,Yellow-crowned Parrot,Amazona,ochrocephala,19003,Aves,[],"[[653.78784, 663.9353, 650.81067, 643.06396, 6..."
28560,ywcpar,[''],[''],ywcpar/iNat78624.ogg,iNat,0.0,8.9918,-79.4877,Amazona ochrocephala,Yellow-crowned Parrot,Amazona,ochrocephala,19003,Aves,[],"[[768.743, 835.997, 849.70135, 833.42285, 816...."
28561,ywcpar,[''],[''],ywcpar/iNat789234.ogg,iNat,0.0,9.2316,-70.2041,Amazona ochrocephala,Yellow-crowned Parrot,Amazona,ochrocephala,19003,Aves,[],"[[867.61707, 895.4965, 894.82715, 899.5896, 90..."
28562,ywcpar,[''],[''],ywcpar/iNat819873.ogg,iNat,0.0,10.5838,-66.8545,Amazona ochrocephala,Yellow-crowned Parrot,Amazona,ochrocephala,19003,Aves,[],"[[337.58582, 337.58582, 337.58582, 337.58582, ..."
